# Experiment 0: Baseline (SmolLM2-135M + LoRA)

This notebook runs the full Crucible pipeline from the experiment config and visualizes training and evaluation results.

**Config:** `conf/experiments/experiment_0_baseline.yaml`
- Data: gbharti/finance-alpaca (5k sample)
- Model: HuggingFaceTB/SmolLM2-135M, LoRA r=8
- Outputs: `outputs/experiment_0_baseline/`

In [ ]:
# Setup: find repo root (parent of notebooks/ when cwd is notebooks/) and run from there
import os
import sys
from pathlib import Path

_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Run Flyte tasks locally (no remote)
os.environ["FLYTE_LOCAL_CACHE_ENABLED"] = "false"

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from crucible.config import CrucibleConfig
from crucible.pipeline import run_full_pipeline

: 

## 1. Run the full pipeline

Set `RUN_PIPELINE = True` to execute data → fine-tune → evaluation. Set to `False` to only load and visualize existing outputs.

In [ ]:
CONFIG_PATH = REPO_ROOT / "conf" / "experiments" / "experiment_0_baseline.yaml"
RUN_PIPELINE = True  # Set False to skip running and only load/visualize existing outputs

# Load config once so we can derive paths whether we run or not
config = CrucibleConfig.from_yaml(CONFIG_PATH)
output_dir = Path(config.finetuning.training.output_dir)
log_file_from_config = Path(config.finetuning.training.log_file)
eval_file_from_config = Path(config.evaluation.output_file)

if RUN_PIPELINE:
    result = run_full_pipeline(config_path=CONFIG_PATH)
    print("Pipeline finished.")
    print(f"  Model: {result.finetune.model_path}")
    print(f"  Train loss: {result.finetune.train_loss:.4f}")
    print(f"  Eval loss: {result.finetune.eval_loss:.4f}")
    print(f"  Training log: {result.finetune.log_file}")
    print(f"  Evaluation: {result.evaluation.output_file}")
else:
    result = None
    print("Skipping pipeline run. Loading from:", log_file_from_config, eval_file_from_config)

╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ in <module>:11                                                                                                  │
│                                                                                                                 │
│ ❱ 11 │   result = run_full_pipeline(config_path=CONFIG_PATH)                                                    │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/crucible/pipeline.py:61 in run_full_pipeline                             │
│                                                                                                                 │
│ ❱ 61 │   return full_pipeline(config_path=str(config_path))  # type: ignore[no-any-return]                      │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/python3.12/site-packages/flytekit/core/base_task.py:373 in     │
│ __call__                                                                                                        │
│                                                                                                                 │
│ ❱ 373 │   │   return flyte_entity_call_handler(self, *args, **kwargs)  # type: ignore                           │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/python3.12/site-packages/flytekit/core/promise.py:1536 in      │
│ flyte_entity_call_handler                                                                                       │
│                                                                                                                 │
│ ❱ 1536 │   │   │   result = cast(LocallyExecutable, entity).local_execute(child_ctx, **kwargs)                  │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/python3.12/site-packages/flytekit/core/base_task.py:353 in     │
│ local_execute                                                                                                   │
│                                                                                                                 │
│ ❱ 353 │   │   │   outputs_literal_map = self.sandbox_execute(ctx, input_literal_map)                            │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/python3.12/site-packages/flytekit/core/base_task.py:430 in     │
│ sandbox_execute                                                                                                 │
│                                                                                                                 │
│ ❱ 430 │   │   return self.dispatch_execute(ctx, input_literal_map)                                              │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/python3.12/site-packages/flytekit/core/base_task.py:767 in     │
│ dispatch_execute                                                                                                │
│                                                                                                                 │
│ ❱ 767 │   │   │   │   │   native_outputs = self.execute(**native_inputs)                                        │
│                                                                                                                 │
│ /Users/jackdicintio/workspace/crucible/.venv/lib/pytho

: 

In [ ]:
# Paths to artifacts (from result or from config when we skipped the run)
OUTPUT_DIR = (
    output_dir if "output_dir" in dir() else REPO_ROOT / "outputs" / "experiment_0_baseline"
)
LOG_FILE = Path(result.finetune.log_file) if result else log_file_from_config
EVAL_FILE = Path(result.evaluation.output_file) if result else eval_file_from_config

## 2. Training log: loss and metrics

Parse the JSONL training log and plot loss (and eval loss / learning rate when present).

In [ ]:
def load_training_log(path: Path) -> pd.DataFrame:
    """Parse JSONL training log into a DataFrame."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


if LOG_FILE.exists():
    train_log = load_training_log(LOG_FILE)
    display(train_log.head(10))
else:
    train_log = pd.DataFrame()
    print("No training log found at", LOG_FILE)

In [ ]:
if not train_log.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss vs step (train loss: 'loss' or 'train_loss')
    ax = axes[0]
    if "loss" in train_log.columns:
        ax.plot(train_log["step"], train_log["loss"], label="Train loss", color="C0")
    if "train_loss" in train_log.columns:
        ax.plot(train_log["step"], train_log["train_loss"], label="Train loss", color="C0")
    if "eval_loss" in train_log.columns:
        eval_steps = train_log.dropna(subset=["eval_loss"])["step"]
        eval_vals = train_log.dropna(subset=["eval_loss"])["eval_loss"]
        if len(eval_steps):
            ax.plot(eval_steps, eval_vals, label="Eval loss", color="C1", marker="o", markersize=4)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Training & evaluation loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Learning rate vs step (if logged)
    ax = axes[1]
    if "learning_rate" in train_log.columns:
        ax.plot(train_log["step"], train_log["learning_rate"], color="C2")
        ax.set_xlabel("Step")
        ax.set_ylabel("Learning rate")
        ax.set_title("Learning rate schedule")
    else:
        ax.text(
            0.5, 0.5, "No learning_rate in log", ha="center", va="center", transform=ax.transAxes
        )
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training log to plot.")

## 3. Evaluation metrics

Load the evaluation results JSON and plot quantitative metrics (ROUGE, BERTScore, semantic similarity, refusal, etc.).

In [ ]:
if EVAL_FILE.exists():
    with open(EVAL_FILE) as f:
        eval_results = json.load(f)
    quantitative = eval_results.get("quantitative", {})
    display(pd.DataFrame(quantitative).T.round(4))
else:
    quantitative = {}
    print("No evaluation results at", EVAL_FILE)

In [ ]:
if quantitative:
    # Flatten metric name -> value for bar chart (one subplot per metric group or one combined)
    flat = []
    for metric_name, scores in quantitative.items():
        for key, val in scores.items():
            flat.append({"metric": metric_name, "sub_metric": key, "value": val})
    df_flat = pd.DataFrame(flat)

    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(df_flat))
    colors = plt.cm.tab10(df_flat["metric"].astype("category").cat.codes % 10)
    ax.bar(x, df_flat["value"], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{row.metric}\n{row.sub_metric}" for _, row in df_flat.iterrows()],
        rotation=45,
        ha="right",
    )
    ax.set_ylabel("Score")
    ax.set_title("Evaluation metrics (Experiment 0 baseline)")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No quantitative metrics to plot.")

## 4. Qualitative samples (if present)

If the evaluation was run with a qualitative prompt set, show a few example inputs and model outputs.

In [ ]:
if EVAL_FILE.exists():
    with open(EVAL_FILE) as f:
        eval_results = json.load(f)
    qual = eval_results.get("qualitative", [])
    if qual:
        n_show = min(5, len(qual))
        for i, item in enumerate(qual[:n_show]):
            print(f"--- Sample {i + 1} ---")
            inp = str(item.get("input", "") or "")
            out = str(item.get("output", "") or "")
            short_inp = inp[:200] + ("..." if len(inp) > 200 else "")
            short_out = out[:300] + ("..." if len(out) > 300 else "")
            print("Input:", short_inp)
            print("Output:", short_out)
            print()
    else:
        print("No qualitative results in evaluation output.")
else:
    print("No evaluation file to load.")

## 5. Summary

- **Training log:** `outputs/experiment_0_baseline/training_log.jsonl`  
- **Evaluation:** `outputs/experiment_0_baseline/evaluation_results.json`  
- **Model checkpoint:** `outputs/experiment_0_baseline/`  
- **Tracking:** Run metadata is stored in the configured backend (e.g. `experiments.db`) when tracking is enabled in the config.